# Part 2: Predicting Customer Churn with Logistic Regression

**Goal:** Predict whether a customer will churn (cancel / leave) based on their age, usage, spending, customer-service calls, tenure, contract type, and region.

**Model:** Logistic Regression (scikit-learn)

**Why logistic regression?** Churn is a yes/no outcome, so this is a binary classification problem. Logistic regression outputs a probability of churn, and its coefficients show which factors push risk up or down.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 42

## 2. Create the dataset

I generated 200 synthetic customers. The chance that a customer churns is driven by factors that are known churn drivers in real subscription businesses:

- **Month-to-month contracts** churn more than one- or two-year contracts.
- **More customer-service calls** → higher churn risk.
- **Longer tenure** and **higher usage** → lower churn risk.
- Customers **under 25** are slightly more likely to churn.

`region` and `purchase_amount` were deliberately given **no** effect, so I can check whether the model correctly treats them as unimportant.

In [2]:
def generate_churn_data(n_records=200):
    rng = np.random.default_rng(RANDOM_SEED)
    age = rng.integers(18, 75, size=n_records)
    monthly_usage = rng.normal(50, 20, size=n_records).clip(1, 150)
    purchase_amount = rng.normal(80, 30, size=n_records).clip(5, 300)
    customer_service_calls = rng.poisson(2, size=n_records)
    tenure_months = rng.integers(1, 72, size=n_records)
    region = rng.choice(["North", "South", "East", "West"], size=n_records)
    contract_type = rng.choice(["Month-to-month", "One year", "Two year"], size=n_records, p=[0.55, 0.25, 0.20])

    contract_risk = np.select(
        [contract_type == "Month-to-month", contract_type == "One year", contract_type == "Two year"],
        [0.9, 0.0, -0.5])
    logit = (-0.7 + contract_risk + 0.25 * customer_service_calls - 0.04 * tenure_months
             - 0.02 * monthly_usage + 0.3 * (age < 25) + rng.normal(0, 0.4, size=n_records))
    churn = (rng.uniform(0, 1, size=n_records) < 1 / (1 + np.exp(-logit))).astype(int)

    return pd.DataFrame({"age": age, "monthly_usage": monthly_usage.round(1),
                         "purchase_amount": purchase_amount.round(2),
                         "customer_service_calls": customer_service_calls,
                         "tenure_months": tenure_months, "contract_type": contract_type,
                         "region": region, "churn": churn})

df = generate_churn_data()
print("Dataset shape:", df.shape)
print(f"Churn rate: {df['churn'].mean():.1%}")
df.head()

Dataset shape: (200, 8)
Churn rate: 17.5%


,age,monthly_usage,purchase_amount,customer_service_calls,tenure_months,contract_type,region,churn
0,23,58.0,27.30,3,13,One year,North,0
1,62,31.9,89.84,1,31,Two year,South,0
2,55,42.4,131.82,4,15,Month-to-month,East,1
3,43,76.0,33.98,3,4,Month-to-month,North,0
4,42,42.9,105.91,0,27,Two year,North,0


**Important:** only about 1 in 6 customers churns, so the classes are **imbalanced**. This matters a lot for evaluation (see step 4).

## 3. Prepare features and split the data

- **Numeric features** are standardized (mean 0, std 1) so they're on the same scale and their coefficients can be compared.
- **Categorical features** (`region`, `contract_type`) are one-hot encoded.
- The train/test split is **stratified** so both sets keep the same churn rate.

In [3]:
numeric_features = ["age", "monthly_usage", "purchase_amount", "customer_service_calls", "tenure_months"]
categorical_features = ["region", "contract_type"]
X = df[numeric_features + categorical_features]
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(drop="first"), categorical_features),
])

## 4. First attempt: default logistic regression

In [4]:
baseline = Pipeline([("preprocess", preprocessor),
                     ("classifier", LogisticRegression(max_iter=1000))])
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred_base):.3f}\n")
print(classification_report(y_test, y_pred_base))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, y_pred_base))

Accuracy: 0.850

              precision    recall  f1-score   support

           0       0.85      1.00      0.92        33
           1       1.00      0.14      0.25         7

    accuracy                           0.85        40
   macro avg       0.92      0.57      0.58        40
weighted avg       0.87      0.85      0.80        40

Confusion matrix [[TN, FP], [FN, TP]]:
[[33  0]
 [ 6  1]]


**Why this result is misleading:** accuracy is 85%, which sounds good, but look at the churn class (row `1`). **Recall is only 0.14**, meaning the model catches just 1 of the 7 customers who actually churned. Because most customers don't churn, the model scores well by predicting "no churn" almost every time. For a business, missing churners is the costly mistake, so this model isn't useful as-is.

## 5. Improved model: balanced class weights

Setting `class_weight="balanced"` tells the model to weight mistakes on the rare class (churners) more heavily during training.

In [5]:
model = Pipeline([("preprocess", preprocessor),
                  ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}\n")
print(classification_report(y_test, y_pred))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.575

              precision    recall  f1-score   support

           0       0.90      0.55      0.68        33
           1       0.25      0.71      0.37         7

    accuracy                           0.57        40
   macro avg       0.57      0.63      0.52        40
weighted avg       0.79      0.57      0.63        40

Confusion matrix [[TN, FP], [FN, TP]]:
[[18 15]
 [ 2  5]]


**Result:** Recall for churners jumps from 0.14 to **0.71** (5 of 7 churners caught instead of 1). The trade-off is more false alarms: precision drops to 0.25, and overall accuracy falls to about 58%, because the model now flags 15 loyal customers as at-risk.

For churn, this trade-off is usually worth it. Sending a retention offer to a customer who wasn't going to leave costs little; losing a customer you could have saved costs a lot. **This shows why accuracy alone is the wrong metric for imbalanced problems.** I use this balanced model for the rest of the notebook.

## 6. Predict churn for a new customer

A 30-year-old on a month-to-month contract, low usage, 5 customer-service calls, and only 3 months of tenure.

In [6]:
new_customer = pd.DataFrame({"age": [30], "monthly_usage": [15], "purchase_amount": [45],
                             "customer_service_calls": [5], "tenure_months": [3],
                             "contract_type": ["Month-to-month"], "region": ["South"]})
prob = model.predict_proba(new_customer)[0][1]
print(f"Churn probability: {prob:.2f} -> {'AT RISK' if prob >= 0.5 else 'not at risk'}")

Churn probability: 0.95 -> AT RISK


This customer has almost every risk factor, so a high churn probability makes sense. In practice, they'd be a good candidate for a retention offer (e.g., a discount for switching to an annual contract).

## 7. Interpret the coefficients

Because the numeric features are standardized, coefficient sizes can be compared. Positive = increases churn risk; negative = decreases it.

In [7]:
ohe = model.named_steps["preprocess"].named_transformers_["cat"]
names = numeric_features + list(ohe.get_feature_names_out(categorical_features))
coefs = pd.Series(model.named_steps["classifier"].coef_[0], index=names).round(3)
coefs.sort_values().to_frame("coefficient")

,coefficient
contract_type_Two year,-1.521
contract_type_One year,-1.018
tenure_months,-0.921
age,-0.339
monthly_usage,-0.244
region_South,-0.173
region_West,-0.056
purchase_amount,0.089
region_North,0.136
customer_service_calls,0.539


**Takeaways:**

- **Two-year and one-year contracts** have the largest negative coefficients: long contracts are the strongest protection against churn.
- **Tenure** is strongly negative: longer-standing customers are less likely to leave.
- **Customer-service calls** is the strongest positive driver: frustrated customers churn.
- **Region** coefficients are small, which matches how the data was built (region had no real effect). This is a good sanity check.

## 8. Limitations

- The test set has only 40 customers (about 7 churners), so these metrics would move noticeably with a different split. Cross-validation would give a more stable estimate.
- The data is synthetic, so the model is recovering relationships I built in. Real churn data (e.g., the IBM Telco dataset) would be noisier.